# Day 13 — Customer-Level Treatment and Exposure Investigation

## Project

EconoCausal — Dynamic Pricing via Double Machine Learning

## Objectives

* Investigate customer, campaign, and coupon relationships.
* Examine whether customer-level campaign exposure can be established.
* Validate identifiers across the available datasets.
* Distinguish recorded associations from verified treatment assignment.
* Document the evidence and limitations relevant to causal inference.

## Expected Outcome

A preliminary assessment of whether the available datasets provide sufficient evidence to identify customer-level campaign exposure.


In [1]:
import pandas as pd
import numpy as np

# Load the required datasets
train = pd.read_csv("../data/raw/train.csv")

campaigns = pd.read_csv(
    "../data/raw/campaign_data.csv"
)

customers = pd.read_csv(
    "../data/raw/customer_demographics.csv"
)

coupon_item_mapping = pd.read_csv(
    "../data/raw/coupon_item_mapping.csv"
)

transactions = pd.read_csv(
    "../data/raw/customer_transaction_data.csv"
)

# Display dataset dimensions
print("===== DATASET DIMENSIONS =====")
print("Train:", train.shape)
print("Campaigns:", campaigns.shape)
print("Customers:", customers.shape)
print("Coupon-item mapping:", coupon_item_mapping.shape)
print("Transactions:", transactions.shape)

===== DATASET DIMENSIONS =====
Train: (78369, 5)
Campaigns: (28, 4)
Customers: (760, 7)
Coupon-item mapping: (92663, 2)
Transactions: (1324566, 7)


In [2]:
# Inspect key columns in the training dataset
print("===== TRAIN DATASET =====")
print(train.columns.tolist())
display(train.head())

# Inspect campaign information
print("\n===== CAMPAIGN DATASET =====")
print(campaigns.columns.tolist())
display(campaigns.head())

# Inspect customer information
print("\n===== CUSTOMER DATASET =====")
print(customers.columns.tolist())
display(customers.head())

# Inspect coupon-item relationships
print("\n===== COUPON-ITEM MAPPING =====")
print(coupon_item_mapping.columns.tolist())
display(coupon_item_mapping.head())

===== TRAIN DATASET =====
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']


,id,campaign_id,coupon_id,customer_id,redemption_status
0,1,13,27,1053,0
1,2,13,116,48,0
2,6,9,635,205,0
3,7,13,644,1050,0
4,9,8,1017,1489,0



===== CAMPAIGN DATASET =====
['campaign_id', 'campaign_type', 'start_date', 'end_date']


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13



===== CUSTOMER DATASET =====
['customer_id', 'age_range', 'marital_status', 'rented', 'family_size', 'no_of_children', 'income_bracket']


,customer_id,age_range,marital_status,rented,family_size,no_of_children,income_bracket
0,1,70+,Married,0,2,NaN,4
1,6,46-55,Married,0,2,NaN,5
2,7,26-35,NaN,0,3,1,3
3,8,26-35,NaN,0,4,2,6
4,10,46-55,Single,0,1,NaN,5



===== COUPON-ITEM MAPPING =====
['coupon_id', 'item_id']


,coupon_id,item_id
0,105,37
1,107,75
2,494,76
3,522,77
4,518,77


In [3]:
# Validate customer, campaign, and coupon identifiers

print("===== IDENTIFIER VALIDATION =====")

# Check required columns
required_columns = {
    "train": ["customer_id", "campaign_id", "coupon_id"],
    "campaigns": ["campaign_id"],
    "customers": ["customer_id"],
    "coupon_item_mapping": ["coupon_id"]
}

datasets = {
    "train": train,
    "campaigns": campaigns,
    "customers": customers,
    "coupon_item_mapping": coupon_item_mapping
}

for dataset_name, columns in required_columns.items():
    for col in columns:
        if col not in datasets[dataset_name].columns:
            raise KeyError(
                f"Column '{col}' not found in {dataset_name}. "
                f"Available columns: {datasets[dataset_name].columns.tolist()}"
            )

# Validate campaign IDs
campaign_match = train["campaign_id"].isin(
    campaigns["campaign_id"]
)

print("\nCampaign ID matches:")
print("Matched records:", int(campaign_match.sum()))
print("Unmatched records:", int((~campaign_match).sum()))

# Validate customer IDs
customer_match = train["customer_id"].isin(
    customers["customer_id"]
)

print("\nCustomer ID matches:")
print("Matched records:", int(customer_match.sum()))
print("Unmatched records:", int((~customer_match).sum()))

# Validate coupon IDs
coupon_match = train["coupon_id"].isin(
    coupon_item_mapping["coupon_id"]
)

print("\nCoupon ID matches:")
print("Matched records:", int(coupon_match.sum()))
print("Unmatched records:", int((~coupon_match).sum()))

===== IDENTIFIER VALIDATION =====

Campaign ID matches:
Matched records: 78369
Unmatched records: 0

Customer ID matches:
Matched records: 43661
Unmatched records: 34708

Coupon ID matches:
Matched records: 78369
Unmatched records: 0


In [4]:
# Analyze customer-campaign relationships

customer_campaign = (
    train.groupby("customer_id")
    .agg(
        campaign_count=("campaign_id", "nunique"),
        coupon_count=("coupon_id", "nunique"),
        total_records=("customer_id", "size")
    )
    .reset_index()
)

print("===== CUSTOMER-CAMPAIGN SUMMARY =====")
display(customer_campaign.head(10))

print("\n===== SUMMARY STATISTICS =====")
display(
    customer_campaign[
        ["campaign_count", "coupon_count", "total_records"]
    ].describe().round(2)
)

# Check whether customers appear in multiple campaigns
print("\nCustomers appearing in multiple campaigns:",
      (customer_campaign["campaign_count"] > 1).sum())

print("Customers appearing in one campaign:",
      (customer_campaign["campaign_count"] == 1).sum())

===== CUSTOMER-CAMPAIGN SUMMARY =====


,customer_id,campaign_count,coupon_count,total_records
0,1,4,93,94
1,3,2,42,42
2,4,1,23,23
3,5,3,52,52
4,6,2,32,32
5,7,7,118,125
6,8,6,93,98
7,9,2,43,43
8,10,2,36,36
9,11,4,70,72



===== SUMMARY STATISTICS =====


,campaign_count,coupon_count,total_records
count,1428.00,1428.00,1428.00
mean,2.98,53.99,54.88
std,1.67,28.81,29.90
min,1.00,8.00,8.00
25%,2.00,32.00,33.00
50%,3.00,50.00,50.00
75%,4.00,71.25,73.00
max,11.00,187.00,199.00



Customers appearing in multiple campaigns: 1116
Customers appearing in one campaign: 312


In [5]:
# Check duplicate customer-campaign-coupon combinations

combination_cols = [
    "customer_id",
    "campaign_id",
    "coupon_id"
]

duplicate_combinations = train.duplicated(
    subset=combination_cols,
    keep=False
)

print("===== DUPLICATE COMBINATION ANALYSIS =====")

print("Total training records:", len(train))

print(
    "Duplicate customer-campaign-coupon records:",
    duplicate_combinations.sum()
)

print(
    "Unique customer-campaign-coupon combinations:",
    train[combination_cols].drop_duplicates().shape[0]
)

print(
    "Customers with repeated customer-campaign-coupon combinations:",
    train.loc[duplicate_combinations, "customer_id"].nunique()
)

# Display examples, if any duplicates exist
if duplicate_combinations.any():
    display(
        train.loc[duplicate_combinations, combination_cols]
        .sort_values(combination_cols)
        .head(20)
    )
else:
    print("No duplicate combinations found.")

===== DUPLICATE COMBINATION ANALYSIS =====
Total training records: 78369
Duplicate customer-campaign-coupon records: 24
Unique customer-campaign-coupon combinations: 78357
Customers with repeated customer-campaign-coupon combinations: 12


,customer_id,campaign_id,coupon_id
14850,15,13,450
24627,15,13,450
37669,389,10,689
64294,389,10,689
46146,565,13,726
75782,565,13,726
71759,626,5,689
73994,626,5,689
32378,659,29,443
59747,659,29,443


In [6]:
# Day 13: Customer-Level Exposure Assessment

assessment = pd.DataFrame([
    {
        "check": "Customer identifier validation",
        "finding": "Customer IDs were checked against customer demographics",
        "causal_implication": "Identifier matching does not establish treatment exposure"
    },
    {
        "check": "Campaign identifier validation",
        "finding": "Campaign IDs were checked against campaign data",
        "causal_implication": "Campaign linkage alone does not establish customer assignment"
    },
    {
        "check": "Coupon identifier validation",
        "finding": "Coupon IDs were checked against coupon-item mapping",
        "causal_implication": "Coupon mapping does not prove a customer received a coupon"
    },
    {
        "check": "Customer-campaign relationships",
        "finding": "Customer campaign and coupon counts were analyzed",
        "causal_implication": "Observed relationships are descriptive, not proof of exposure"
    },
    {
        "check": "Duplicate combination analysis",
        "finding": (
            f"{duplicate_combinations.sum()} records belong to "
            "repeated customer-campaign-coupon combinations"
        ),
        "causal_implication": "Repeated combinations require interpretation; they do not establish treatment"
    },
    {
        "check": "Verified treatment assignment",
        "finding": "Not established by the relationship checks alone",
        "causal_implication": "Additional evidence of assignment or exposure is required for causal analysis"
    }
])

display(assessment)

# Save the assessment
assessment.to_csv(
    "../reports/day13_customer_exposure_assessment.csv",
    index=False
)

print("Assessment saved successfully.")

,check,finding,causal_implication
0,Customer identifier validation,Customer IDs were checked against customer dem...,Identifier matching does not establish treatme...
1,Campaign identifier validation,Campaign IDs were checked against campaign data,Campaign linkage alone does not establish cust...
2,Coupon identifier validation,Coupon IDs were checked against coupon-item ma...,Coupon mapping does not prove a customer recei...
3,Customer-campaign relationships,Customer campaign and coupon counts were analyzed,"Observed relationships are descriptive, not pr..."
4,Duplicate combination analysis,24 records belong to repeated customer-campaig...,Repeated combinations require interpretation; ...
5,Verified treatment assignment,Not established by the relationship checks alone,Additional evidence of assignment or exposure ...


Assessment saved successfully.
